# 03 — PSE/OPQ Tokenization

Train and evaluate PSE (Parallel Subspace Encoding) via FAISS OPQ/PQ (from DiffGRM).

In [1]:
DRY_RUN = False

In [2]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from src.data.yambda_loader import load_embeddings
from src.tokenizers.pse_opq import PSETokenizer
from src.evaluation.metrics import (
    reconstruction_mse, codebook_utilization, collision_rate, entropy_per_level
)

In [3]:
with open("../configs/default.yaml") as f:
    config = yaml.safe_load(f)

pse_cfg = config["pse_opq"]
suffix = "_dryrun" if DRY_RUN else ""

# Dry-run overrides
n_digit = min(4, pse_cfg["n_digit"]) if DRY_RUN else pse_cfg["n_digit"]
codebook_size = min(32, pse_cfg["codebook_size"]) if DRY_RUN else pse_cfg["codebook_size"]
pca_dim = min(32, pse_cfg["pca_dim"]) if DRY_RUN else pse_cfg["pca_dim"]

print(f"n_digit={n_digit}, codebook_size={codebook_size}, pca_dim={pca_dim}")

n_digit=10, codebook_size=512, pca_dim=120


## Load Embeddings

In [4]:
embed_path = config["data"]["output_path"].replace(".parquet", f"{suffix}.parquet")
item_ids, embeddings = load_embeddings(embed_path)

Loaded 500000 embeddings of dim 128 from data/embeddings.parquet


## PCA Variance Analysis

In [5]:
from sklearn.decomposition import PCA
from pathlib import Path

n_components = min(embeddings.shape[1], 50 if DRY_RUN else 256)
pca_analysis = PCA(n_components=n_components)
pca_analysis.fit(embeddings)

cumvar = np.cumsum(pca_analysis.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(cumvar) + 1), cumvar, "o-", markersize=3)
ax.set_xlabel("Number of Components")
ax.set_ylabel("Cumulative Explained Variance")
ax.set_title("PCA Variance Analysis")
ax.axhline(y=0.95, color="r", linestyle="--", label="95% variance")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

plot_dir = Path(f"../outputs/pse_opq")
plot_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(plot_dir / f"pca_variance{suffix}.png", dpi=150)
print(f"Plot saved to {plot_dir / f'pca_variance{suffix}.png'}")

plt.show()

print(f"Variance at {pca_dim} components: {cumvar[min(pca_dim, len(cumvar))-1]:.4f}")

Plot saved to ../outputs/pse_opq/pca_variance.png
Variance at 120 components: 0.9996


## Train PSE/OPQ

In [6]:
model = PSETokenizer(
    n_digit=n_digit,
    codebook_size=codebook_size,
    pca_dim=pca_dim,
    disable_opq=pse_cfg["disable_opq"],
    use_gpu=pse_cfg["use_gpu"],
)
model.fit(embeddings)

Applying PCA: 128 -> 120


Training FAISS index (n_digit=10, codebook_size=512, dim=120)...


Index trained and populated with 500000 vectors


## Encode & Evaluate

In [7]:
sids = model.encode(embeddings)
reconstructed = model.reconstruct(embeddings)

print(f"SIDs shape: {sids.shape}")
print(f"Reconstruction MSE: {reconstruction_mse(embeddings, reconstructed):.6f}")
print(f"Collision rate: {collision_rate(sids):.6f}")

SIDs shape: (500000, 10)
Reconstruction MSE: 0.000830


Collision rate: 0.026098


In [8]:
# Code frequency per subspace
fig, axes = plt.subplots(1, min(4, n_digit), figsize=(14, 3))
if n_digit == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    ax.hist(sids[:, i], bins=codebook_size, edgecolor="black")
    ax.set_title(f"Subspace {i} codes")
    ax.set_xlabel("Code")
plt.tight_layout()

fig.savefig(plot_dir / f"pse_code_histograms{suffix}.png", dpi=150)
print(f"Plot saved to {plot_dir / f'pse_code_histograms{suffix}.png'}")

plt.show()

Plot saved to ../outputs/pse_opq/pse_code_histograms.png


In [9]:
# Per-level stats
util = codebook_utilization(sids, codebook_size)
ent = entropy_per_level(sids, codebook_size)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
levels = np.arange(n_digit)

axes[0].bar(levels, util, color="coral")
axes[0].set_title("Codebook Utilization per Subspace")
axes[0].set_xlabel("Subspace")
axes[0].set_ylim(0, 1.05)

axes[1].bar(levels, ent, color="coral")
axes[1].set_title("Normalized Entropy per Subspace")
axes[1].set_xlabel("Subspace")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()

fig.savefig(plot_dir / f"pse_utilization_entropy{suffix}.png", dpi=150)
print(f"Plot saved to {plot_dir / f'pse_utilization_entropy{suffix}.png'}")

plt.show()

Plot saved to ../outputs/pse_opq/pse_utilization_entropy.png


## Save Results

In [10]:
from pathlib import Path

# Save SIDs
sids_path = pse_cfg["sids_path"].replace(".parquet", f"{suffix}.parquet")
sids_df = pd.DataFrame({"item_id": item_ids})
for i in range(sids.shape[1]):
    sids_df[f"sid_{i}"] = sids[:, i]
Path(sids_path).parent.mkdir(parents=True, exist_ok=True)
sids_df.to_parquet(sids_path, index=False)
print(f"SIDs saved to {sids_path}")

# Save index
index_path = pse_cfg["index_path"].replace(".faiss", f"{suffix}.faiss")
model.save(index_path)

SIDs saved to outputs/pse_opq/sids.parquet
PSE index saved to outputs/pse_opq/index.faiss
